# Explore a run

Pick any saved run by name. The notebook auto-dispatches to the correct `Predictor` based on `config.pipeline`, loads the dataset's `val` split (also from `config`), runs inference, and shows source / ground-truth / prediction side by side.

Same notebook works for UNet and SAM-head (and any future pipeline that registers a `Predictor` in `pato.experiments.load_predictor`).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from config import paths
from pato.dataset import DatasetViewer
from pato.experiments import list_runs, load_run, load_predictor
from pato.visualize import load_image, show_side_by_side

## 1. Pick a run

In [ ]:
list_runs(paths.runs)

In [ ]:
RUN_NAME = list_runs(paths.runs)[-1]   # ← change to any name from the list above
run_path = paths.runs / RUN_NAME
run = load_run(run_path)
print(f"Run        : {run.name}")
print(f"Pipeline   : {run.config.get('pipeline')}")
print(f"Best ckpt  : {run.best_checkpoint().name if run.best_checkpoint() else '(none)'}")
print(f"Dataset    : {run.config.get('dataset_root')}")
run.config

## 2. Load the predictor (auto-dispatched on `pipeline`)

In [ ]:
predictor = load_predictor(run_path)
type(predictor).__name__

## 3. Load the val split and predict on a few samples

In [ ]:
val = DatasetViewer(root=Path(run.config["dataset_root"]), split="val")
print(f"val split: {len(val)} samples")

In [ ]:
sample_indices = [0, len(val) // 2, len(val) - 1]
for idx in sample_indices:
    sample = val[idx]
    predicted = predictor.predict(sample)
    print(f"--- {val.sample_ids[idx]} — image {sample.image.shape[:2]} ---")
    fig = show_side_by_side(
        sample.image, sample.mask, predicted,
        titles=["source", "ground truth", f"{type(predictor).__name__} prediction"],
        mask_zmax=11,
    )
    fig.show()

## 4. Predict on an arbitrary image path

Change `image_path` to any image file you want — no ground truth needed.

In [ ]:
image_path = paths.nmsc_5x / "Images" / "BCC_1.tif"   # ← change me

image = load_image(image_path)
predicted = predictor.predict(image_path)
print(f"image: {image.shape}, prediction: {predicted.shape} {predicted.dtype}")

show_side_by_side(
    image, predicted,
    titles=["source", f"{type(predictor).__name__} prediction"],
    mask_zmax=11,
)